# LLM 실습 과제 - 내풀이 v1

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv('/home/pc/dev/metacode/MetacodeAssessment/.env')
print(f"API Key loaded: {'Yes' if os.environ.get('OPENAI_API_KEY') else 'No'}")

In [1]:
# =========================
# 문제 1 (난이도: 하)
# =========================
# 텍스트를 토큰화하고 → ID로 변환 → 다시 토큰으로 복원

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
text = "LLM is powerful"
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)
decoded_tokens = tokenizer.convert_ids_to_tokens(token_ids)

print("[문제1]")
print(f"tokens: {tokens}")
print(f"token_ids: {token_ids}")
print(f"decoded_tokens: {decoded_tokens}")

/home/pc/dev/metacode/MetacodeAssessment/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[문제1]
tokens: ['ll', '##m', 'is', 'powerful']
token_ids: [2222, 2213, 2003, 3928]
decoded_tokens: ['ll', '##m', 'is', 'powerful']


In [2]:
# =========================
# 문제 2 (난이도: 하)
# =========================
# Scaled Dot-Product Attention 구현

import torch
import torch.nn.functional as F

Q = torch.randn(2, 4)
K = torch.randn(2, 4)
V = torch.randn(2, 4)

d = Q.size(-1)

scores = torch.matmul(Q, K.t()) / (d ** 0.5)
attention_weights = F.softmax(scores, dim=-1)
output = torch.matmul(attention_weights, V)

print("[문제2]")
print(f"scores:\n{scores}")
print(f"attention_weights:\n{attention_weights}")
print(f"output:\n{output}")

[문제2]
scores:
tensor([[ 0.5231,  0.1541],
        [ 0.4893, -1.9328]])
attention_weights:
tensor([[0.5912, 0.4088],
        [0.9185, 0.0815]])
output:
tensor([[ 1.6801, -1.2668,  0.0732,  0.3790],
        [ 1.7656, -1.3127,  1.0061,  0.7942]])


In [3]:
# =========================
# 문제 3 (난이도: 상)
# =========================
# KV Cache 성능 효과 실험

import time
import torch

def attention_compute(x):
    return torch.matmul(x, x.T)


def run_no_cache(seq_len=50):
    x = torch.randn(seq_len, 16)
    start = time.time()
    for i in range(1, seq_len + 1):
        # 매번 처음부터 i까지 전체 재계산
        _ = attention_compute(x[:i])
    elapsed = time.time() - start
    return elapsed


def run_with_cache(seq_len=50):
    x = torch.randn(seq_len, 16)
    cache = []
    start = time.time()
    for i in range(seq_len):
        # 새 토큰 하나만 계산하고 캐시에 저장
        new_token = x[i].unsqueeze(0)  # (1, 16)
        result = attention_compute(new_token)
        cache.append(result)
    elapsed = time.time() - start
    return elapsed, len(cache)


t1 = run_no_cache()
t2, cache_size = run_with_cache()

print("[문제3]")
print(f"No Cache: {t1:.6f}s")
print(f"With Cache: {t2:.6f}s")
print(f"Cache size: {cache_size}")
print(f"Speedup: {t1/t2:.2f}x")

[문제3]
No Cache: 0.003227s
With Cache: 0.000716s
Cache size: 50
Speedup: 4.51x


In [4]:
# =========================
# 문제 4 (난이도: 중)
# =========================
# LLM 메시지 구조 작성

messages = []

messages.append({"role": "system", "content": "You are a helpful assistant."})
messages.append({"role": "user", "content": "What is machine learning?"})
messages.append({"role": "assistant", "content": "Machine learning is a subset of AI that enables systems to learn from data."})

print("[문제4]")
for msg in messages:
    print(f"[{msg['role']}] {msg['content']}")

[문제4]
[system] You are a helpful assistant.
[user] What is machine learning?
[assistant] Machine learning is a subset of AI that enables systems to learn from data.


In [5]:
# =========================
# 문제 5 (난이도: 중)
# =========================
# Tool schema 정의 및 함수 실행

import json

def get_weather(city):
    return f"{city} sunny"

# tool schema 정의
tool_schema = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the weather for a given city",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "The city name"}
            },
            "required": ["city"]
        }
    }
}

# JSON arguments 파싱 및 함수 실행
tool_call_args = '{"city": "Seoul"}'
parsed_args = json.loads(tool_call_args)
result = get_weather(**parsed_args)

print("[문제5]")
print(f"Tool schema: {tool_schema['function']['name']}")
print(f"Parsed args: {parsed_args}")
print(f"Result: {result}")

[문제5]
Tool schema: get_weather
Parsed args: {'city': 'Seoul'}
Result: Seoul sunny


In [6]:
# =========================
# 문제 6 (난이도: 중)
# =========================
# Tool router 구현

def add(a, b):
    return a + b

def multiply(a, b):
    return a * b

def tool_router(name, args):
    if name == "add":
        return add(**args)
    elif name == "multiply":
        return multiply(**args)
    else:
        return f"Unknown tool: {name}"

print("[문제6]")
print(f"add(3, 5) = {tool_router('add', {'a': 3, 'b': 5})}")
print(f"multiply(4, 7) = {tool_router('multiply', {'a': 4, 'b': 7})}")

[문제6]
add(3, 5) = 8
multiply(4, 7) = 28


In [23]:
# =========================
# 문제 7 (난이도: 상)
# =========================
# Tool Calling 전체 흐름 구현
# ※ 실행하려면 OPENAI_API_KEY 환경변수 설정 필요

from openai import OpenAI
import json
import os

import os
api_key = os.environ.get("OPENAI_API_KEY")

if not api_key:
    print("[문제7] OPENAI_API_KEY 미설정 → 스킵 (코드 로직은 위 셀 참고)")
else:
    client = OpenAI(api_key=api_key)

    def calculator(a, b):
        return a + b

    messages = [{"role": "user", "content": "10 + 20 계산해줘"}]

    tools = [
        {
            "type": "function",
            "function": {
                "name": "calculator",
                "description": "Add two numbers",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "a": {"type": "number"},
                        "b": {"type": "number"}
                    },
                    "required": ["a", "b"]
                }
            }
        }
    ]

    # 1차 LLM 호출 → tool_call 추출
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools
    )

    assistant_msg = response.choices[0].message
    messages.append(assistant_msg)

    # tool_call이 있으면 함수 실행 후 결과를 메시지에 추가
    if assistant_msg.tool_calls:
        for tool_call in assistant_msg.tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)
            result = calculator(**func_args)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(result)
            })

        # 2차 LLM 호출 → 최종 응답
        final_response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
            tools=tools
        )
        final_answer = final_response.choices[0].message.content
    else:
        final_answer = assistant_msg.content

    print("[문제7]")
    print(f"Final answer: {final_answer}")

[문제7]
Final answer: 10 + 20의 계산 결과는 30입니다.


In [24]:
# =========================
# 문제 8 (난이도: 상)
# =========================
# Streaming 응답 처리
# ※ 실행하려면 OPENAI_API_KEY 환경변수 설정 필요
import os
api_key = os.environ.get("OPENAI_API_KEY")

if not api_key:
    print("[문제8] OPENAI_API_KEY 미설정 → 스킵 (코드 로직은 위 셀 참고)")
else:
    client = OpenAI(api_key=api_key)

    cache = []
    MAX_TOKENS = 100

    stream = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": "Explain AI"}],
        stream=True
    )

    collected_text = ""
    token_count = 0

    for chunk in stream:
        delta = chunk.choices[0].delta
        if delta.content:
            token = delta.content
            collected_text += token
            cache.append(token)
            token_count += 1
            print(token, end="", flush=True)

            if token_count >= MAX_TOKENS:
                break

    print()
    print("\n[문제8]")
    print(f"Total tokens collected: {token_count}")
    print(f"Cache length: {len(cache)}")
    print(f"Collected text (first 200 chars): {collected_text[:200]}")

AI, or Artificial Intelligence, refers to the simulation of human intelligence in machines that are programmed to think, learn, and perform tasks typically requiring human cognitive functions. These tasks include understanding natural language, recognizing patterns, solving problems, and making decisions.

There are different types of AI:

1. **Narrow AI (Weak AI):** Designed to perform a specific task, such as voice assistants (e.g., Siri, Alexa), recommendation systems, or image recognition software. Narrow AI operates under a limited

[문제8]
Total tokens collected: 100
Cache length: 100
Collected text (first 200 chars): AI, or Artificial Intelligence, refers to the simulation of human intelligence in machines that are programmed to think, learn, and perform tasks typically requiring human cognitive functions. These t


In [17]:
# =========================
# 문제 9 (난이도: 상)
# =========================
# 외부 API를 활용한 환율 조회

import requests

BASE_URL = "https://open.er-api.com/v6/latest/"

def get_exchange_rate(base, target):
    try:
        response = requests.get(f"{BASE_URL}{base}")
        response.raise_for_status()
        data = response.json()

        if data.get("result") != "success":
            return f"API error: {data.get('error-type', 'unknown')}"

        rates = data["rates"]
        if target not in rates:
            return f"Target currency '{target}' not found"

        return rates[target]
    except requests.RequestException as e:
        return f"Request failed: {e}"

print("[문제9]")
rate = get_exchange_rate("USD", "KRW")
print(f"USD → KRW: {rate}")
rate2 = get_exchange_rate("EUR", "JPY")
print(f"EUR → JPY: {rate2}")

[문제9]
USD → KRW: 1472.46758
EUR → JPY: 187.350453


In [18]:
# =========================
# 문제 10 (난이도: 상)
# =========================
# 자연어 수식 계산 agent

expr = "3 더하기 5 곱하기 2"

# 연산자 변환
expr = expr.replace("더하기", "+").replace("곱하기", "*")
expr = expr.replace("빼기", "-").replace("나누기", "/")
parts = expr.split()

# 순차 계산 (왼쪽에서 오른쪽으로)
result = float(parts[0])
i = 1
while i < len(parts):
    op = parts[i]
    num = float(parts[i + 1])
    if op == "+":
        result += num
    elif op == "-":
        result -= num
    elif op == "*":
        result *= num
    elif op == "/":
        result /= num
    i += 2

print("[문제10]")
print(f"Expression: 3 더하기 5 곱하기 2")
print(f"Parsed: {parts}")
print(f"Result: {result}")

[문제10]
Expression: 3 더하기 5 곱하기 2
Parsed: ['3', '+', '5', '*', '2']
Result: 16.0
